# 🎨 IndusCraft — All-in-One Standalone SDXL LoRA Training Notebook

This notebook contains **100% self-contained Python code** for training both **Workflow A (Multi-View Design Generation)** and **Workflow B (Regional Garment Inpainting)** on Google Colab / Tesla T4 without requiring any GitHub repository clones or external script files.

## 🛠️ 1. Install Required Dependencies

In [ ]:
# Upgrade peft, torchao, and diffusers stack for Colab compatibility
!pip install -q -U torchao peft diffusers transformers accelerate datasets huggingface_hub pillow tqdm bitsandbytes

## ⚙️ 2. Imports & GPU Environment Check

In [ ]:
import os
import gc
import json
import math
import logging
from pathlib import Path
from tqdm.notebook import tqdm

# Prevent CUDA memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageDraw

from transformers import AutoTokenizer, CLIPTextModel, CLIPTextModelWithProjection
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, StableDiffusionXLPipeline, StableDiffusionXLInpaintPipeline
from peft import LoraConfig
from datasets import load_dataset

# Enable TF32 for high performance & numerical stability
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Check CUDA GPU Availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚡ PyTorch Device: {device}')
if torch.cuda.is_available():
    print(f'🎮 GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 📦 3. Self-Contained Dataset Loader & SDXL Dual Prompt Encoder

In [ ]:
class IndusCraftDataset(Dataset):
    """Dataset loader pulling directly from Hugging Face Hub with configuration fallback."""
    def __init__(self, hf_repo: str, config_name: str = 'craft_reference', split: str = 'train', resolution: int = 768):
        self.resolution = resolution
        self.transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
        print(f"Loading dataset from Hugging Face: {hf_repo}...")
        try:
            self.dataset = load_dataset(hf_repo, name=config_name, split=split)
        except Exception:
            print(f"Config '{config_name}' fallback to default split '{split}'...")
            self.dataset = load_dataset(hf_repo, split=split)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        tensor_image = self.transform(image)
        text = item.get('text', item.get('instruction', 'Indian craft design'))
        return {'pixel_values': tensor_image, 'caption': text}

def encode_sdxl_prompts(tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2, prompts, device):
    """Encodes prompts for SDXL dual text encoders (CLIP ViT-L + OpenCLIP ViT-bigG)."""
    with torch.no_grad():
        tokens_1 = tokenizer_1(prompts, padding='max_length', max_length=tokenizer_1.model_max_length, truncation=True, return_tensors='pt').input_ids.to(device)
        prompt_embeds_1 = text_encoder_1(tokens_1, output_hidden_states=True).hidden_states[-2]

        tokens_2 = tokenizer_2(prompts, padding='max_length', max_length=tokenizer_2.model_max_length, truncation=True, return_tensors='pt').input_ids.to(device)
        enc_out_2 = text_encoder_2(tokens_2, output_hidden_states=True)
        prompt_embeds_2 = enc_out_2.hidden_states[-2]
        pooled_prompt_embeds = enc_out_2.text_embeds

        prompt_embeds = torch.cat([prompt_embeds_1, prompt_embeds_2], dim=-1)
    return prompt_embeds, pooled_prompt_embeds

## 🎨 4. WORKFLOW A: Stage 1 — Train Craft Knowledge & Multi-View UNet LoRA
Teaches SDXL authentic Indian craft aesthetics (`Chikankari`, `Kalamkari`, `Phulkari`, `Ajrakh`, etc.) and multi-view prompts (`[front view]`, `[detail view]`, `[pattern view]`).

In [ ]:
# Hyperparameters (Tesla T4 / 15GB VRAM Optimized)
HF_DATASET_REPO = 'vedantjadhav701/induscraft-dataset'
BASE_MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
OUTPUT_DIR_STAGE1 = './induscraft_stage1_craft_lora'

RESOLUTION = 768
BATCH_SIZE = 1
GRAD_ACCUM = 4
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
LORA_RANK = 32
LORA_ALPHA = 32

os.makedirs(OUTPUT_DIR_STAGE1, exist_ok=True)
dtype = torch.float16

# 1. Load Tokenizers & Text Encoders
print('Loading SDXL Tokenizers & Text Encoders...')
tokenizer_1 = AutoTokenizer.from_pretrained(BASE_MODEL, subfolder='tokenizer', use_fast=False)
tokenizer_2 = AutoTokenizer.from_pretrained(BASE_MODEL, subfolder='tokenizer_2', use_fast=False)
text_encoder_1 = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder='text_encoder', torch_dtype=dtype).to(device)
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(BASE_MODEL, subfolder='text_encoder_2', torch_dtype=dtype).to(device)
text_encoder_1.eval()
text_encoder_2.eval()

# 2. Load VAE & UNet
print('Loading VAE & UNet Base Model...')
vae = AutoencoderKL.from_pretrained(BASE_MODEL, subfolder='vae', torch_dtype=dtype).to(device)
vae.eval()

unet = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder='unet', torch_dtype=dtype)
unet.enable_gradient_checkpointing()
noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder='scheduler')

# 3. Configure PEFT LoRA via Diffusers Native API
print(f'Injecting LoRA layers (rank={LORA_RANK}, alpha={LORA_ALPHA})...')
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=['to_k', 'to_q', 'to_v', 'to_out.0'],
    lora_dropout=0.05,
    bias='none',
)
unet.add_adapter(lora_config)
unet.to(device)
trainable_params = [p for p in unet.parameters() if p.requires_grad]
print(f'Trainable LoRA Parameters: {sum(p.numel() for p in trainable_params):,}')

# 4. Prepare Dataset & DataLoader
dataset = IndusCraftDataset(HF_DATASET_REPO, config_name='craft_reference', split='train', resolution=RESOLUTION)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=True)

# 5. Execute Training Loop
gc.collect()
torch.cuda.empty_cache()
print(f'⚡ Starting Stage 1 UNet LoRA Training for {NUM_EPOCHS} Epochs...')
global_step = 0
for epoch in range(NUM_EPOCHS):
    unet.train()
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch in progress_bar:
        images = batch['pixel_values'].to(device, dtype=dtype)
        prompts = batch['caption']

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * vae.config.scaling_factor

        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        prompt_embeds, pooled_prompt_embeds = encode_sdxl_prompts(
            tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2, prompts, device
        )

        add_time_ids = torch.tensor([[RESOLUTION, RESOLUTION, 0, 0, RESOLUTION, RESOLUTION]], dtype=dtype, device=device).repeat(bsz, 1)
        added_cond_kwargs = {'text_embeds': pooled_prompt_embeds.to(dtype), 'time_ids': add_time_ids}

        with torch.amp.autocast('cuda', dtype=dtype):
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states=prompt_embeds.to(dtype), added_cond_kwargs=added_cond_kwargs).sample
            loss = F.mse_loss(model_pred.float(), noise.float(), reduction='mean') / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (global_step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        global_step += 1
        progress_bar.set_postfix({'loss': f'{loss.item() * GRAD_ACCUM:.4f}'})

# Save Trained Stage 1 LoRA Weights
stage1_final_dir = os.path.join(OUTPUT_DIR_STAGE1, 'induscraft_stage1_final')
unet.save_attn_procs(stage1_final_dir)
print(f"🎉 Stage 1 Craft Knowledge LoRA complete! Weights saved to '{stage1_final_dir}'")

# Clean up Stage 1 memory before Stage 2
del unet
del optimizer
gc.collect()
torch.cuda.empty_cache()

## 🧥 5. WORKFLOW B: Stage 2 — Train Regional Garment Inpainting LoRA
Teaches targeted regional craft editing on Collar, Sleeves, Cuffs, Hemline, and Chest while preserving the original garment.

In [ ]:
INPAINT_MODEL = 'diffusers/stable-diffusion-xl-1.0-inpainting-0.1'
OUTPUT_DIR_STAGE2 = './induscraft_stage2_inpainting_lora'
os.makedirs(OUTPUT_DIR_STAGE2, exist_ok=True)

# Thorough GPU memory cleanup before loading Inpainting UNet
if 'unet' in locals():
    del unet
if 'optimizer' in locals():
    del optimizer
gc.collect()
torch.cuda.empty_cache()

print('Loading SDXL Inpainting UNet Model...')
unet_inpaint = UNet2DConditionModel.from_pretrained(INPAINT_MODEL, subfolder='unet', torch_dtype=dtype)
unet_inpaint.enable_gradient_checkpointing()
unet_inpaint.add_adapter(lora_config)
unet_inpaint.to(device)
trainable_inpaint_params = [p for p in unet_inpaint.parameters() if p.requires_grad]
print(f'Trainable Inpainting LoRA Parameters: {sum(p.numel() for p in trainable_inpaint_params):,}')

dataset_inpaint = IndusCraftDataset(HF_DATASET_REPO, config_name='garment_application', split='train', resolution=RESOLUTION)
dataloader_inpaint = DataLoader(dataset_inpaint, batch_size=BATCH_SIZE, shuffle=True)
optimizer_inpaint = torch.optim.AdamW(trainable_inpaint_params, lr=LEARNING_RATE)
scaler_inpaint = torch.amp.GradScaler('cuda', enabled=True)

print(f'⚡ Starting Stage 2 Regional Garment Inpainting LoRA Training for {NUM_EPOCHS} Epochs...')
global_step = 0
for epoch in range(NUM_EPOCHS):
    unet_inpaint.train()
    progress_bar = tqdm(dataloader_inpaint, desc=f'Inpainting Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch in progress_bar:
        images = batch['pixel_values'].to(device, dtype=dtype)
        prompts = batch['caption']
        bsz = images.shape[0]

        masks = torch.zeros((bsz, 1, RESOLUTION, RESOLUTION), device=device, dtype=dtype)
        masks[:, :, :int(RESOLUTION*0.3), int(RESOLUTION*0.3):int(RESOLUTION*0.7)] = 1.0
        masked_images = images * (1.0 - masks)

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * vae.config.scaling_factor
            masked_latents = vae.encode(masked_images).latent_dist.sample() * vae.config.scaling_factor
            resized_masks = F.interpolate(masks, size=latents.shape[-2:], mode='nearest')

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        latent_model_input = torch.cat([noisy_latents, resized_masks, masked_latents], dim=1)
        prompt_embeds, pooled_prompt_embeds = encode_sdxl_prompts(
            tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2, prompts, device
        )

        add_time_ids = torch.tensor([[RESOLUTION, RESOLUTION, 0, 0, RESOLUTION, RESOLUTION]], dtype=dtype, device=device).repeat(bsz, 1)
        added_cond_kwargs = {'text_embeds': pooled_prompt_embeds.to(dtype), 'time_ids': add_time_ids}

        with torch.amp.autocast('cuda', dtype=dtype):
            model_pred = unet_inpaint(latent_model_input, timesteps, encoder_hidden_states=prompt_embeds.to(dtype), added_cond_kwargs=added_cond_kwargs).sample
            loss = F.mse_loss(model_pred.float(), noise.float(), reduction='mean') / GRAD_ACCUM

        scaler_inpaint.scale(loss).backward()

        if (global_step + 1) % GRAD_ACCUM == 0:
            scaler_inpaint.unscale_(optimizer_inpaint)
            torch.nn.utils.clip_grad_norm_(trainable_inpaint_params, 1.0)
            scaler_inpaint.step(optimizer_inpaint)
            scaler_inpaint.update()
            optimizer_inpaint.zero_grad()

        global_step += 1
        progress_bar.set_postfix({'loss': f'{loss.item() * GRAD_ACCUM:.4f}'})

stage2_final_dir = os.path.join(OUTPUT_DIR_STAGE2, 'induscraft_stage2_final')
unet_inpaint.save_attn_procs(stage2_final_dir)
print(f"🎉 Stage 2 Garment Inpainting LoRA complete! Saved to '{stage2_final_dir}'")

## 🎨 6. Test Inference & Generate Sample Design Variations

In [ ]:
print('Loading trained LoRA into SDXL Pipeline...')
pipe = StableDiffusionXLPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to('cuda')
if os.path.exists(stage1_final_dir):
    pipe.load_lora_weights(stage1_final_dir)
    print('Loaded Stage 1 IndusCraft LoRA successfully!')

sample_prompts = [
    'A high resolution detailed fashion design photograph, [front view], of authentic chikankari embroidery style on a white modern kurta, Lucknowi shadow work.',
    'A high resolution detailed fashion design photograph, [detail view], macro close-up shot of chikankari floral jaali stitch texture.',
    'A high resolution detailed fashion design photograph, [pattern view], seamless repeating motif layout of authentic kalamkari tree of life print.',
    'A high resolution detailed fashion design photograph, [flat garment], technical flat layout of a paithani silk saree with zari border.'
]

for idx, prompt in enumerate(sample_prompts):
    image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
    filename = f'design_sample_{idx+1}.png'
    image.save(filename)
    print(f'Saved sample: {filename}')